In [143]:
import networkx as nx
from collections import deque
import random
import pandas as pd
import igraph

In [144]:
def BFS(G, first_node):
    queue = deque()
    visited = list()
    Visit(G, first_node, queue, visited)
    for n in G.nodes():
        if n not in visited:
            Visit(G, n, queue, visited)
    return visited

def Visit(G, v, queue: deque, visited):
    queue.append(v)
    visited.append(v)
    while len(queue) != 0:
        k = queue.popleft()
        for n in nx.all_neighbors(G, k):
            if n not in visited:
                queue.append(n)
                visited.append(n)

In [145]:
def Insertionsort(arr):
    num_comp = 0
    for i in range(1, len(arr)):
        t = arr[i]
        j = i - 1
        while j >= 0:
            num_comp += 1
            if arr[j] < t:
                arr[j+1] = arr[j]
                j = j - 1
            else:
                break
        arr[j+1] = t
    return num_comp

In [146]:
def Quicksort(arr, p, r, rng):
    if p < r:
        q, num_comp = Partition(arr, p, r, rng)
        num_comp += Quicksort(arr, p, q-1, rng)
        num_comp += Quicksort(arr, q+1, r, rng)
        return num_comp
    return 0
    
def Partition(arr, p, r, rng):
    num_comp = 0
    
    random_pivot = rng.randint(p, r)
    Swap(arr, r, random_pivot)

    q = p
    for u in range(p, r):
        num_comp += 1
        if arr[u] >= arr[r]:
            Swap(arr, q, u)
            q += 1
    Swap(arr, q, r)
    return q, num_comp
    
def Swap(arr, i, j):
    t = arr[i]
    arr[i] = arr[j]
    arr[j] = t

In [147]:
rng = random.Random(1231)
m = 16
quinhentos = [nx.barabasi_albert_graph(500, m=m, seed=rng) for _ in range(30)]
mil = [nx.barabasi_albert_graph(1000, m=m, seed=rng) for _ in range(30)]
cnc_mil = [nx.barabasi_albert_graph(5000, m=m, seed=rng) for _ in range(30)]

In [148]:

def gera_resultados_grafos(lista_grafos, tamanho, func_centralidade):

    tamanho_amostra = len(lista_grafos)
    resultado = [tamanho, tamanho_amostra]

    comp_insertion = []
    comp_insertionbfs = []
    comp_quicksort = []

    rng_pivos_quicksort = random.Random(3122)
    rng_ordem_aleatoria = random.Random(4412)

    for grafo in lista_grafos:

        centralidade = func_centralidade(grafo)

        no_maior_valor = max(centralidade, key=centralidade.get)
        pre_ordem = BFS(grafo, no_maior_valor)
        insertion_bfs = Insertionsort([centralidade[n] for n in pre_ordem])

        lista_centralidades = list(centralidade.values())
        rng_ordem_aleatoria.shuffle(lista_centralidades) # utilizamos o shuffle para evitar que a ordem original do grafo (gerado pelo algoritmo barabasi) tenha impacto no experimento
        insertion = Insertionsort(lista_centralidades[:])
        quick = Quicksort(lista_centralidades[:], 0, len(grafo)-1, rng_pivos_quicksort)
        comp_insertion.append(insertion/len(grafo))
        comp_quicksort.append(quick/len(grafo))
        comp_insertionbfs.append(insertion_bfs/len(grafo))

    s_insertion = pd.Series(comp_insertion)
    s_insertionbfs = pd.Series(comp_insertionbfs)
    s_quicksort = pd.Series(comp_quicksort)
    resultado.extend([
        s_insertion.mean(),
        s_insertion.std(),
        s_insertionbfs.mean(),
        s_insertionbfs.std(),
        s_quicksort.mean(),
        s_quicksort.std(),
    ])

    return resultado



In [149]:
resultados_grau = pd.DataFrame([], columns=['Nº de vértices', 
                                            'Nº de grafos', 
                                            'M Insertion Sort', 
                                            'DP Insertion Sort', 
                                            'M Insertion Sort com BFS',
                                            'DP Insertion Sort com BFS', 
                                            'M Quicksort', 
                                            'DP Quicksort'
])


r_quinhentos = gera_resultados_grafos(quinhentos, '500', nx.degree_centrality)
resultados_grau.loc[len(resultados_grau)] = r_quinhentos
r_mil = gera_resultados_grafos(mil, '1000', nx.degree_centrality)
resultados_grau.loc[len(resultados_grau)] = r_mil
r_c_mil = gera_resultados_grafos(cnc_mil, '5000', nx.degree_centrality)
resultados_grau.loc[len(resultados_grau)] = r_c_mil
resultados_grau.round(2)

,Nº de vértices,Nº de grafos,M Insertion Sort,DP Insertion Sort,M Insertion Sort com BFS,DP Insertion Sort com BFS,M Quicksort,DP Quicksort
0,500,30,119.80,3.39,73.81,2.86,17.52,0.93
1,1000,30,240.62,4.79,159.38,4.23,29.28,0.71
2,5000,30,1190.26,10.87,876.48,9.89,126.59,0.95


In [150]:
resultados_proximidade = pd.DataFrame([], columns=['Nº de vértices', 
                                            'Nº de grafos', 
                                            'M Insertion Sort', 
                                            'DP Insertion Sort', 
                                            'M Insertion Sort com BFS',
                                            'DP Insertion Sort com BFS', 
                                            'M Quicksort', 
                                            'DP Quicksort'
])

def closeness(G):
    ig = igraph.Graph.from_networkx(G)
    closeness = ig.closeness(normalized=True)

    d = dict()
    for v in range(ig.vcount()):
        d[ig.vs[v]['_nx_name']] = closeness[v]
    return d

r_quinhentos = gera_resultados_grafos(quinhentos, '500', closeness)
resultados_proximidade.loc[len(resultados_proximidade)] = r_quinhentos
r_mil = gera_resultados_grafos(mil, '1000', closeness)
resultados_proximidade.loc[len(resultados_proximidade)] = r_mil
r_c_mil = gera_resultados_grafos(cnc_mil, '5000', closeness)
resultados_proximidade.loc[len(resultados_proximidade)] = r_c_mil
resultados_proximidade.round(2)

,Nº de vértices,Nº de grafos,M Insertion Sort,DP Insertion Sort,M Insertion Sort com BFS,DP Insertion Sort com BFS,M Quicksort,DP Quicksort
0,500,30,124.64,3.56,70.84,3.48,9.78,0.71
1,1000,30,251.63,4.81,147.46,5.63,11.13,0.62
2,5000,30,1247.98,12.14,715.22,16.51,14.56,0.81


In [151]:
resultados_intermediacao = pd.DataFrame([], columns=['Nº de vértices', 
                                            'Nº de grafos', 
                                            'M Insertion Sort', 
                                            'DP Insertion Sort', 
                                            'M Insertion Sort com BFS',
                                            'DP Insertion Sort com BFS', 
                                            'M Quicksort', 
                                            'DP Quicksort'
])

def betweenness(G):
    ig = igraph.Graph.from_networkx(G)
    closeness = ig.betweenness(normalized=True)

    d = dict()
    for v in range(ig.vcount()):
        d[ig.vs[v]['_nx_name']] = closeness[v]
    return d


r_quinhentos = gera_resultados_grafos(quinhentos, '500', betweenness)
resultados_intermediacao.loc[len(resultados_intermediacao)] = r_quinhentos
r_mil = gera_resultados_grafos(mil, '1000', betweenness)
resultados_intermediacao.loc[len(resultados_intermediacao)] = r_mil
r_c_mil = gera_resultados_grafos(cnc_mil, '5000', betweenness)
resultados_intermediacao.loc[len(resultados_intermediacao)] = r_c_mil
resultados_intermediacao.round(2)

,Nº de vértices,Nº de grafos,M Insertion Sort,DP Insertion Sort,M Insertion Sort com BFS,DP Insertion Sort com BFS,M Quicksort,DP Quicksort
0,500,30,125.36,3.28,84.57,3.55,9.52,0.75
1,1000,30,251.98,5.13,175.47,4.71,10.92,0.48
2,5000,30,1250.13,11.29,870.54,10.86,14.10,0.64


In [152]:
resultados_autovetor = pd.DataFrame([], columns=['Nº de vértices', 
                                            'Nº de grafos', 
                                            'M Insertion Sort', 
                                            'DP Insertion Sort', 
                                            'M Insertion Sort com BFS',
                                            'DP Insertion Sort com BFS', 
                                            'M Quicksort', 
                                            'DP Quicksort'
])

def func(G):
    return nx.eigenvector_centrality(G, max_iter=500)

r_quinhentos = gera_resultados_grafos(quinhentos, '500', func)
resultados_autovetor.loc[len(resultados_autovetor)] = r_quinhentos
r_mil = gera_resultados_grafos(mil, '1000', func)
resultados_autovetor.loc[len(resultados_autovetor)] = r_mil
r_c_mil = gera_resultados_grafos(cnc_mil, '5000', func)
resultados_autovetor.loc[len(resultados_autovetor)] = r_c_mil
resultados_autovetor.round(2)

,Nº de vértices,Nº de grafos,M Insertion Sort,DP Insertion Sort,M Insertion Sort com BFS,DP Insertion Sort com BFS,M Quicksort,DP Quicksort
0,500,30,125.25,3.46,69.90,3.50,9.72,0.61
1,1000,30,252.27,4.78,146.08,6.34,11.22,0.82
2,5000,30,1248.39,12.28,698.65,18.87,14.22,0.66
